# 01 - Ingestie in HDFS

Scop: incarcam fisierul `flights.csv` din `/data` in HDFS si verificam ca Spark il poate citi distribuit.

Acest notebook nu salveaza Parquet. Pentru laptop slab este mai stabil sa pastram raw CSV in HDFS si sa salvam ulterior CSV curatat.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("FlightsProject")
    .master("spark://master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 21:43:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8
Master: spark://master:7077


## Configurare cai

Datasetul descarcat de pe Kaggle trebuie pus in folderul `/data` din containerul master. In Docker Compose, acest folder este legat de folderul local `./data`.


In [2]:
import os
import subprocess

LOCAL_CSV = "/data/flights.csv"
HDFS_RAW_DIR = "/flights/raw"
HDFS_RAW_CSV = "hdfs://master:9000/flights/raw/flights.csv"

print("Local CSV exists:", os.path.exists(LOCAL_CSV), LOCAL_CSV)


Local CSV exists: True /data/flights.csv


## Copiere in HDFS

Comenzile HDFS se ruleaza din notebook. Daca fisierul exista deja in HDFS, este inlocuit.


In [3]:
if not os.path.exists(LOCAL_CSV):
    raise FileNotFoundError("Nu exista /data/flights.csv. Pune datasetul Kaggle in folderul ./data de pe host.")

cmds = [
    ["hdfs", "dfs", "-mkdir", "-p", HDFS_RAW_DIR],
    ["hdfs", "dfs", "-rm", "-f", f"{HDFS_RAW_DIR}/flights.csv"],
    ["hdfs", "dfs", "-put", LOCAL_CSV, f"{HDFS_RAW_DIR}/flights.csv"],
    ["hdfs", "dfs", "-ls", HDFS_RAW_DIR],
]

for cmd in cmds:
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0 and "No such file" not in result.stderr:
        print(result.stderr)


$ hdfs dfs -mkdir -p /flights/raw

$ hdfs dfs -rm -f /flights/raw/flights.csv

$ hdfs dfs -put /data/flights.csv /flights/raw/flights.csv

$ hdfs dfs -ls /flights/raw
Found 1 items
-rw-r--r--   2 root supergroup   42858599 2026-04-26 21:44 /flights/raw/flights.csv



## Citire cu Spark

Spark citeste direct din HDFS. `repartition(4)` foloseste mai bine cei doi workeri, dar ramane rezonabil pentru resurse mici.


In [4]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(HDFS_RAW_CSV)
    .repartition(4)
)

print("Numar coloane:", len(df.columns))
print("Numar randuri:", df.count())
df.printSchema()
df.show(5, truncate=False)


Numar coloane: 21


Numar randuri: 336776
root
 |-- id: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: double (nullable = true)
 |-- sched_dep_time: integer (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- arr_time: double (nullable = true)
 |-- sched_arr_time: integer (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: double (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)
 |-- time_hour: timestamp (nullable = true)
 |-- name: string (nullable = true)



[Stage 8:============================================>              (3 + 1) / 4]

+------+----+-----+---+--------+--------------+---------+--------+--------------+---------+-------+------+-------+------+----+--------+--------+----+------+-------------------+----------------------+
|id    |year|month|day|dep_time|sched_dep_time|dep_delay|arr_time|sched_arr_time|arr_delay|carrier|flight|tailnum|origin|dest|air_time|distance|hour|minute|time_hour          |name                  |
+------+----+-----+---+--------+--------------+---------+--------+--------------+---------+-------+------+-------+------+----+--------+--------+----+------+-------------------+----------------------+
|144104|2013|3    |9  |1320.0  |1325          |-5.0     |1450.0  |1505          |-15.0    |MQ     |4475  |N713MQ |LGA   |RDU |69.0    |431     |13  |25    |2013-03-09 13:00:00|Envoy Air             |
|104950|2013|12   |24 |1513.0  |1520          |-7.0     |1630.0  |1705          |-35.0    |AA     |341   |N542AA |LGA   |ORD |118.0   |733     |15  |20    |2013-12-24 15:00:00|American Airlines Inc.|


## Verificare partitionare

Nu fortam operatii grele. Doar verificam cate partitii are DataFrame-ul.


In [5]:
print("Numar partitii:", df.rdd.getNumPartitions())
print("Coloane:", df.columns)


[Stage 11:=============================>                            (2 + 2) / 4]

Numar partitii: 4
Coloane: ['id', 'year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour', 'name']


[Stage 11:===========================================>              (3 + 1) / 4]

## Concluzie

Fisierul raw este in HDFS si poate fi citit de Spark. Urmatorul notebook face EDA, cleanup si salveaza varianta curatata ca CSV in HDFS, nu Parquet.
